# 04 — RAG Pipeline

RAG pipeline demo: document loading, chunking, embedding, and LLM Q&A.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv()

print("Add PDF files to data/documents/ before running this notebook")
print("Recommended: EIA STEO, OPEC Monthly Oil Market Report, IEA Oil Market Report")

## 1. Load Documents

In [ ]:
from src.rag.document_loader import DocumentLoader
loader = DocumentLoader(documents_dir='../data/documents')
pages = loader.load_directory()
print(f"Loaded {len(pages)} pages from PDF documents")
if pages:
    print(f"First page preview: {pages[0]['text'][:200]}...")

## 2. Chunk Text

In [ ]:
from src.rag.chunker import TextChunker
chunker = TextChunker(chunk_size=1000, chunk_overlap=200)
if pages:
    chunks = chunker.chunk_pages(pages)
    print(f"Created {len(chunks)} text chunks")
    print(f"Average chunk size: {sum(c['char_count'] for c in chunks) / len(chunks):.0f} chars")

## 3. Generate Embeddings

In [ ]:
from src.rag.embeddings import EmbeddingGenerator
# Uses sentence-transformers (free, local)
embedder = EmbeddingGenerator(provider='sentence-transformers')
if pages:
    chunks_with_embs = embedder.embed_chunks(chunks[:10])  # Demo with 10 chunks
    print(f"Embedding dimension: {len(chunks_with_embs[0]['embedding'])}")

## 4. Build Vector Store and Query

In [ ]:
from src.rag.vector_store import VectorStoreManager
store = VectorStoreManager(store_path='../chroma_db')
if pages:
    store.add_chunks(chunks_with_embs)
    
    # Query
    query = "What factors drive crude oil price volatility?"
    query_emb = embedder.embed_query(query)
    results = store.query(query_emb, top_k=3)
    for i, r in enumerate(results):
        print(f"Result {i+1} (score={r['score']:.3f}):")
        print(f"  Source: {r['filename']} p.{r['page_number']}")
        print(f"  Text: {r['text'][:200]}...")
        print()